# 07 — Wafer Chip Misalignment Detection (Alignment)

## Objective

Measure die placement errors in semiconductor packaging: register a test image against a reference pattern and report translation offsets plus statistical process-control metrics.

## What you'll see

- A synthetic reference pattern vs. a shifted test image
- Template matching (match score) and registration (translation offsets) from `TemplateMatcher` / `RegistrationAnalyzer`
- An SPC summary (control-chart style metrics) from `SPCAnalyzer`

The goal is to make the core registration and alignment workflow feel tangible and editable.

## How to use this notebook

Change the parameters in the next cell and rerun the later cells to see how the results change.

## How to use this notebook

- Edit the parameters in the next cell to change the shift or rotation.
- Run the cells in order so you can see how each stage changes the result.
- Compare how a small shift behaves versus a larger one.
- Use the printed metrics as a guide to understand what changed.

In [ ]:
# Editable parameters: change these values and rerun the notebook.
shift = 3
rotation = 2.0

print("Configuration:")
print(f"  shift={shift}")
print(f"  rotation={rotation}")

In [ ]:
import numpy as np

from optical_metrology.analysis import RegistrationAnalyzer, SPCAnalyzer, TemplateMatcher
from optical_metrology.detector import DigitalImage

In [ ]:
size = 64
rng = np.random.default_rng(7)
yy, xx = np.mgrid[:size, :size]
reference = np.full((size, size), 0.12, dtype=float)

center = size / 2
wafer_radius = size * 0.3
reference += 0.22 * np.exp(-((yy - center) ** 2 + (xx - center) ** 2) / (2 * wafer_radius ** 2))

for cy in (size // 4, size // 2, size * 3 // 4):
    for cx in (size // 4, size // 2, size * 3 // 4):
        die_mask = ((xx - cx) ** 2 + (yy - cy) ** 2) <= 20
        reference[die_mask] = 0.9
        reference[np.abs(xx - cx) < 2] = 0.25
        reference[np.abs(yy - cy) < 2] = 0.25

missing_dies = {(size // 2, size // 4), (size * 3 // 4, size * 3 // 4)}
for cy, cx in missing_dies:
    reference[np.abs(xx - cx) < 8] = 0.08
    reference[np.abs(yy - cy) < 8] = 0.08

reference[np.abs(xx - center) < 2] = 0.55
reference[np.abs(yy - center) < 2] = 0.55
reference += 0.03 * np.sin(0.14 * xx + 0.07 * yy)
reference += rng.normal(0.0, 0.02, size=reference.shape)
reference = np.clip(reference, 0.0, 1.0)

test = reference.copy()
if abs(rotation) > 0:
    try:
        from scipy.ndimage import rotate
        test = rotate(test, angle=float(rotation), mode="nearest", reshape=False)
    except Exception:
        pass
if shift != 0:
    test = np.roll(test, shift=(shift, shift), axis=(0, 1))
test += rng.normal(0.0, 0.015, size=test.shape)
test = np.clip(test, 0.0, 1.0)

ref_image = DigitalImage(pixels=reference, metadata={"bit_depth": 12})
test_image = DigitalImage(pixels=test, metadata={"bit_depth": 12})

print("Created synthetic reference and test images")

In [ ]:
matcher = TemplateMatcher(template=reference[10:22, 10:22])
match_report = matcher.analyze(test_image)
print("Template match:")
for key, value in match_report.measurements.items():
    print(f"  {key}: {value}")

In [ ]:
registration = RegistrationAnalyzer()
registration_report = registration.analyze_pair(ref_image, test_image)
print("Registration summary:")
for key, value in registration_report.measurements.items():
    print(f"  {key}: {value}")

In [ ]:
spc = SPCAnalyzer(usl=2.0, lsl=-2.0, target=0.0, metric="dx")
spc_report = spc.analyse_measurements([
    {"dx": registration_report.measurements["dx"]},
    {"dx": registration_report.measurements["dx"] + 0.2},
    {"dx": registration_report.measurements["dx"] - 0.1},
])
print("SPC summary:")
for key, value in spc_report.measurements.items():
    print(f"  {key}: {value}")

## Try next

Try changing one thing at a time:
- increase the shift to see how the match score changes
- increase the rotation parameter to see how registration behaves
- compare a small shift with a larger one to understand the sensitivity of the methods